# 34 GRB 211211A: the two slices that don't match Gompertz et al.

Notebook 14 compares my E_peak of GRB 211211A with Gompertz et al.'s (2023) slice by slice. 18 of the 20 slices agree within 2 sigma,
but in 2-4 s and 8-10 s mine is 20-27% higher (3.6 and 2.2 sigma). There E_peak is above 1 MeV, so notebook 14 first put it down to the
BGO, which I start at 200 keV and they start at 300. Notebook 16 then tried their start, and it moves both slices up, further from
theirs. So it isn't the BGO start.

This notebook looks for the reason. Each slice is fitted again exactly as in notebook 16, with the same spectra, responses, two-break model
and starting points, and with one thing changed at a time. 6-8 s, between the two and also above 1 MeV, is fitted alongside as a control:
there mine agrees with theirs (1607 against 1543 keV).

1. The response files
2. Their energy ranges, one choice at a time
3. The high-energy tail
4. What this says

What I can't repeat is their joint fit with Swift/BAT, or the exact shape of their own two-break function, since I fit GBM alone with my
two-break model.

The sigmas are worked out as in notebook 14: the difference over the two statistical errors in quadrature. A refit with something
changed has no error search of its own, so it uses the error of notebook 14's fit. Run it from `ignacio/`. It reads GRB 211211A's GBM
data and takes about a minute.

In [1]:
import re
import warnings

import numpy as np
import pandas as pd
from astropy.io import fits

import settings_211211A as S
from spectral_fit_tools import (BandLowBreak, channel_mask, cstat, fit_model, make_spectra, rsp2_path, scaled_starts,
                                window_response)

warnings.filterwarnings("ignore")     # GDT's fitter warns on most trial steps near the parameter limits
pd.set_option("display.width", 200)

# Gompertz et al. (2023) Table 1, the same values as notebook 14: E_peak and its error in keV, and beta. The two slices that disagree,
# and 6-8 s as a control
GOMPERTZ = pd.DataFrame({"Ep": [951.6, 1053.6, 1543.0], "Ep_err": [47.4, 72.0, 47.5], "beta": [-2.74, -2.90, -3.09]},
                        index=["g02_04", "g08_10", "g06_08"])
LABELS = list(GOMPERTZ.index)
DETECTORS = S.SPECTRAL_DETECTORS

windows = pd.read_csv(S.WINDOWS_FILE, comment="#", index_col="label")
table = pd.read_csv(S.WINDOWS_FILE.parent / "epeak_211211A.csv").set_index("label")
WIN = {label: (windows.t_start[label], windows.t_stop[label]) for label in LABELS}
MY_ERR = 0.5 * (table.Epeak_err_lo + table.Epeak_err_hi)      # notebook 14's statistical error, the mean of its two sides

spectra = {d: make_spectra(S.DATA_DIR, S.BURST_ID, d, WIN, S.BKG_WINDOWS, S.BKG_ORDER)[0] for d in DETECTORS}
responses = {(d, label): window_response(S.DATA_DIR, S.BURST_ID, d, WIN[label])[0] for d in DETECTORS for label in LABELS}


def start_vector(label):
    """Notebook 14's two-break fit of the slice, as a starting point (the same as notebook 16)."""
    row = table.loc[label]
    found = re.search(r"break at ([\d.]+) keV, alpha1 (-?[\d.]+) below it, alpha2 (-?[\d.]+) above", row.notes)
    eb, a1, a2 = (float(x) for x in found.groups())
    return [1.0, eb, a1, a2, row.Epeak_keV, min(row.beta, -2.05), 2.0, 100.0]


def window_data(label, nai=S.FIT_RANGE_NAI, bgo=S.FIT_RANGE_BGO, cut=None):
    """Spectra, backgrounds, responses and channel masks of one slice. cut leaves an energy range out of the NaI."""
    rsps = [responses[d, label] for d in DETECTORS]
    masks = []
    for d, r in zip(DETECTORS, rsps):
        mask = channel_mask(r, bgo if d.startswith("b") else nai)
        if cut and not d.startswith("b"):
            mask &= ~channel_mask(r, cut)
        masks.append(mask)
    return [spectra[d][label][0] for d in DETECTORS], [spectra[d][label][1] for d in DETECTORS], rsps, masks


def fit_again(data, start, fix_beta=False):
    """The two-break fit from start, and from E_peak moved either way, as in notebook 16. Returns the parameters and C."""
    function = BandLowBreak()
    if fix_beta:
        function.free = [True, True, True, True, True, False, False, False]
    shapes = [list(start)] + [[v * f if i == 4 else v for i, v in enumerate(start)] for f in (0.8, 1.25)]
    fit = fit_model(*data, function, scaled_starts(function, *data, shapes) + [list(start)], polish=True)
    return list(function.default_values), cstat(fit)


def against_theirs(label, ep):
    """Mine over theirs, and the difference over the two errors in quadrature (notebook 14's sigma)."""
    return ep / GOMPERTZ.Ep[label], (ep - GOMPERTZ.Ep[label]) / np.hypot(MY_ERR[label], GOMPERTZ.Ep_err[label])


table.loc[LABELS, ["model", "Epeak_keV", "beta"]].assign(my_err=MY_ERR[LABELS]).join(GOMPERTZ.add_prefix("Gompertz_"))

,model,Epeak_keV,beta,my_err,Gompertz_Ep,Gompertz_Ep_err,Gompertz_beta
label,,,,,,,
g02_04,BandLowBreak,1212.14,-3.26579,55.91450,951.6,47.4,-2.74
g08_10,BandLowBreak,1257.74,-10.00000,59.43825,1053.6,72.0,-2.90
g06_08,BandLowBreak,1606.91,-3.29694,37.13570,1543.0,47.5,-3.09


## 1. The response files

For GRB 160625B the version of the response files mattered a lot (notebook 33). The archive's files were made again in 2020, after the
papers had been written, and the new ones give a main-emission E_peak 13-56% higher. If that had happened here, Gompertz et al. would
have fitted older responses than the ones I use. Each file's header says when it was made.

In [2]:
for d in DETECTORS:
    path = rsp2_path(S.DATA_DIR, S.BURST_ID, d)
    with fits.open(path) as f:
        print(path.name, "made", f[0].header["DATE"])

glg_cspec_n2_bn211211549_v02.rsp2 made 2021-12-11T19:30:37
glg_cspec_na_bn211211549_v02.rsp2 made 2021-12-11T19:30:37
glg_cspec_b0_bn211211549_v02.rsp2 made 2021-12-11T19:30:37


All three are version 02, made at 19:30 on 11 December 2021, the day of the burst. Any version Gompertz et al. could have used was made that
same day, so a later remake of the responses, as for GRB 160625B, isn't the reason here.

## 2. Their energy ranges, one choice at a time

Gompertz et al. fit the NaI from 10 to 900 keV without 30-40 keV, which is around the iodine K-edge, and the BGO from 300 keV to 40 MeV.
I fit the NaI from 8 to 900 keV and the BGO from 200 keV. Each of their three choices is tried on its own, and then all three together.
The first row of each slice is the fit with my ranges, which should give back notebook 14's E_peak. A change of ranges changes the
channels in the fit, so C can't be compared between rows here.

In [3]:
VARIANTS = {"mine (NaI 8-900, BGO from 200)": {}, "NaI from 10 keV": dict(nai=(10.0, 900.0)), "NaI without 30-40 keV": dict(cut=(30.0, 40.0)),
            "BGO from 300 keV": dict(bgo=(300.0, 40000.0)),
            "all three, their ranges": dict(nai=(10.0, 900.0), cut=(30.0, 40.0), bgo=(300.0, 40000.0))}
base, base_c, rows = {}, {}, []
for label in LABELS:
    base[label], base_c[label] = fit_again(window_data(label), start_vector(label))
    for name, kw in VARIANTS.items():
        p = base[label] if not kw else fit_again(window_data(label, **kw), base[label])[0]
        ratio, sigma = against_theirs(label, p[4])
        rows.append(dict(label=label, variant=name, Epeak=p[4], shift=p[4] - base[label][4], Eb=p[1], beta=p[5], ratio=ratio, sigma=sigma))
ranges = pd.DataFrame(rows).set_index(["label", "variant"])
ranges.round(3)

Epeak   shift      Eb    beta  ratio  sigma
label  variant                                                                       
g02_04 mine (NaI 8-900, BGO from 200)  1212.141   0.000  61.004  -3.269  1.274  3.554
       NaI from 10 keV                 1209.272  -2.869  57.817  -3.255  1.271  3.515
       NaI without 30-40 keV           1231.181  19.040  83.675  -3.369  1.294  3.814
       BGO from 300 keV                1237.706  25.566  59.164  -3.325  1.301  3.903
       all three, their ranges         1248.321  36.180  74.384  -3.395  1.312  4.048
g08_10 mine (NaI 8-900, BGO from 200)  1257.740   0.000  26.145 -10.000  1.194  2.186
       NaI from 10 keV                 1253.487  -4.253  23.286 -10.000  1.190  2.141
       NaI without 30-40 keV           1250.346  -7.394  29.416 -10.000  1.187  2.107
       BGO from 300 keV                1260.207   2.467  26.165 -10.000  1.196  2.213
       all three, their ranges         1246.178 -11.562  25.677 -10.000  1.183  2.063
g06_08 mine (NaI 8-900, BGO from 200)  1606.910   0.000  51.955  -3.302  1.041  1.060
       NaI from 10 keV                 1606.961   0.051  50.963  -3.301  1.041  1.061
       NaI without 30-40 keV           1614.181   7.271  59.049  -3.319  1.046  1.181
       BGO from 300 keV                1614.762   7.852  51.696  -3.312  1.047  1.190
       all three, their ranges         1617.137  10.227  57.075  -3.317  1.048  1.230

With my ranges every slice gives back notebook 14's E_peak exactly, and the BGO from 300 keV gives back notebook 16's shifts (+25.6 keV in
2-4 s and +2.5 keV in 8-10 s), so these are the same fits. Their ranges hardly move E_peak. All three together give 1248 keV in 2-4 s and
1246 keV in 8-10 s, a change of +3% and -1%, so mine stays 31% and 18% above theirs (4.0 and 2.1 sigma). Leaving out 30-40 keV moves the
low-energy break (from 61 to 84 keV in 2-4 s), but hardly the peak. In 6-8 s, the control, their ranges move E_peak by under 1%, and it
still agrees with theirs. So their energy ranges aren't the reason either.

## 3. The high-energy tail

What does differ is the tail above the peak. Their beta is -2.74 in 2-4 s and -2.90 in 8-10 s. Mine is -3.27 in 2-4 s, and in 8-10 s my
two-break fit has no tail at all: beta runs to its limit of -10, which notebook 14's table flags. Near the peak the two trade off. A
steeper tail lets the peak sit higher and still give the counts the BGO sees, so a fit can move E_peak up and beta down together for
little change in C.

So beta is fixed at their value and everything else is fitted again. The rise in C says how much GBM's data alone dislike their tail.

In [4]:
rows = []
for label in LABELS:
    start = list(base[label])
    start[5] = GOMPERTZ.beta[label]
    fixed, c_fixed = fit_again(window_data(label), start, fix_beta=True)
    for name, p, c in (("beta free, as notebook 14", base[label], base_c[label]), ("beta fixed at theirs", fixed, c_fixed)):
        ratio, sigma = against_theirs(label, p[4])
        rows.append(dict(label=label, variant=name, Epeak=p[4], Eb=p[1], alpha2=p[3], beta=p[5], C=c, dC=c - base_c[label], ratio=ratio,
                         sigma=sigma))
tail = pd.DataFrame(rows).set_index(["label", "variant"])
tail.round(3)

Epeak      Eb  alpha2    beta        C     dC  ratio  sigma
label  variant                                                                                  
g02_04 beta free, as notebook 14  1212.141  61.004  -1.327  -3.269  585.034  0.000  1.274  3.554
       beta fixed at theirs       1075.487  49.698  -1.261  -2.740  594.511  9.478  1.130  1.690
g08_10 beta free, as notebook 14  1257.740  26.145  -1.439 -10.000  463.660  0.000  1.194  2.186
       beta fixed at theirs       1142.477  24.394  -1.416  -2.900  471.396  7.735  1.084  0.952
g06_08 beta free, as notebook 14  1606.910  51.955  -1.294  -3.302  691.680  0.000  1.041  1.060
       beta fixed at theirs       1572.599  50.613  -1.286  -3.090  695.396  3.715  1.019  0.491

Fixing beta at theirs brings my E_peak down to 1075 keV in 2-4 s and 1142 keV in 8-10 s, 13% and 8% above theirs, which is 1.7 and 1.0
sigma. That costs 9.5 and 7.7 in C, so GBM's data alone prefer my steeper tail, by about 3 sigma for the one parameter. The control moves
the same way but less (from 1607 to 1573 keV, for 3.7 in C), and it still agrees. So about half of the gap in each slice is the peak trading
off against the tail, and the rest is within the errors.

## 4. What this says

- The two slices where my E_peak is 20-27% above Gompertz et al.'s aren't explained by the BGO start (notebook 16), the response files or
  their energy ranges.
- About half of the gap is the peak trading off against the high-energy tail. With their tail, my E_peak is 13% and 8% above theirs, within
  1.7 and 1.0 sigma.
- GBM's data alone prefer my steeper tail (C lower by 9.5 and 7.7), so notebook 14's values stay as they are. In these two slices E_peak
  carries a systematic of about 10% from the shape fitted above the peak.
- What's left is what I can't repeat here: their joint fit with Swift/BAT and the shape of their own two-break function. Either could set
  their tail.